#### Fine-tune SmolVLM on Visual Question Answering using Consumer GPU with QLoRA

In [ ]:
## installing essential libraires
!pip install -q accelerate datasets peft bitsandbytes tensorboard
## acclerate - for fast cuda performane
## peft - parameter efficient optimization
## bitsandbytes - optimized loading model in 4 bits
## tensorboard - to view the loss curves

In [ ]:
# !pip install flash-attn --no-build-isolation
#  # to get flash attention optimized mechanism

In [ ]:
## this is required to push model directly to hugging face
from huggingface_hub import notebook_login
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
notebook_login()

In [ ]:
## importing model processor
import torch
from transformers import AutoProcessor,BitsAndBytesConfig, Idefics3ForConditionalGeneration
from peft import LoraConfig,prepare_model_for_kbit_training, get_peft_model

USE_LORA = False
USE_QLORA = True
USE_4_BIT = True
SMOL = True

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f'[INFO] DEVICE : {DEVICE}')

In [ ]:
model_id = "HuggingFaceTB/SmolVLM-Base" if SMOL else "HuggingFaceM4/Idefics3-8B-Llama3"

processor = AutoProcessor.from_pretrained(
    model_id
)

if USE_LORA or USE_QLORA:
  lora_config = LoraConfig( # here lora config is set
        r=8,
        lora_alpha=8,
        lora_dropout=0.1,
        target_modules=['down_proj','o_proj','k_proj','q_proj','gate_proj','up_proj','v_proj'],
        use_dora=False if USE_QLORA else True,
        init_lora_weights="gaussian"
    )
  lora_config.inference_mode = False # inference mode is removed to train
  if USE_QLORA :
    bnb_config = BitsAndBytesConfig( # here with "bitsandbytes" library we set the config to get the most optimized 4bit weights of original model
          load_in_4bit=True,
          bnb_4bit_use_double_quant=True,
          bnb_4bit_quant_type="nf4",
          bnb_4bit_compute_dtype=torch.bfloat16
      ) # it is and library to quantize the model in 4bit or 8 bit percision
  model = Idefics3ForConditionalGeneration.from_pretrained(
        model_id,
        quantization_config=bnb_config if USE_QLORA else None,
        _attn_implementation="sdpa", # flash_attention is giving error on colab
        device_map="auto"
    )
  model.add_adapter(lora_config) # lora adaptar added
  model.enable_adapters() # lora adapter enables
  model = prepare_model_for_kbit_training(model) # this library provide extra preprocessing where the raw weights and other thing from 4bit precision loaded model to trainable
  model = get_peft_model(model, lora_config) # model is converted to use peft methods
  print(model.get_nb_trainable_parameters()) # the trainable weights
else:
  model = Idefics3ForConditionalGeneration.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,
        _attn_implementation="sdpa",
  ).to(DEVICE)


In [ ]:
model.model.model

In [ ]:
# if you'd like to only fine-tune LLM ## only llm (output) generation is finetuned not the vision part
for param in model.model.model.vision_model.parameters():
    param.requires_grad = False

In [ ]:
for param in model.parameters():
  print(param)
  break

In [ ]:
from datasets import load_dataset
ds = load_dataset('merve/vqav2-small', trust_remote_code=True)
## loading dataset

In [ ]:
split_ds = ds["validation"].train_test_split(test_size=0.5)
train_ds = split_ds["train"]

In [ ]:
train_ds

In [ ]:
processor

In [ ]:
processor.tokenizer.convert_tokens_to_ids("<image>")

In [ ]:
## collate function
image_token_id = processor.tokenizer.convert_tokens_to_ids("<image>")
def collate_fn(examples):
  texts = []
  images = []
  for example in examples:
      image = example["image"]
      if image.mode != 'RGB':
        image = image.convert('RGB')
      question = example["question"]
      answer = example["multiple_choice_answer"]
      messages = [ # the format what is needed
          {
              "role": "user",
              "content": [
                  {"type": "text", "text": "Answer briefly."},
                  {"type": "image"},
                  {"type": "text", "text": question}
              ]
          },
          {
              "role": "assistant",
              "content": [
                  {"type": "text", "text": answer}
              ]
          }
      ]
      text = processor.apply_chat_template(messages, add_generation_prompt=False) # image token is added
      texts.append(text.strip())
      images.append([image])

  batch = processor(text=texts, images=images, return_tensors="pt", padding=True)
  labels = batch["input_ids"].clone()
  labels[labels == processor.tokenizer.pad_token_id] = -100
  labels[labels == image_token_id] = -100
  batch["labels"] = labels # the output labels are added

  return batch

In [ ]:
from transformers import TrainingArguments, Trainer

model_name = model_id.split("/")[-1]

training_args = TrainingArguments(
    num_train_epochs=1,
    per_device_train_batch_size=2, # batch size optimized
    gradient_accumulation_steps=4,
    warmup_steps=50,
    learning_rate=1e-4,
    weight_decay=0.01,
    logging_steps=25,
    save_strategy="steps",
    save_steps=250,
    save_total_limit=1,
    optim="paged_adamw_8bit", # for 8-bit, keep this, else adamw_hf
    bf16=True, # underlying precision for 8bit
    output_dir=f"./{model_name}-vqav2",
    hub_model_id=f"{model_name}-vqav2",
    report_to="tensorboard",
    remove_unused_columns=False,
    gradient_checkpointing=True
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=collate_fn,
    train_dataset=train_ds,
)
## model is trained

In [ ]:
trainer.train()
## this is good but training time is too long to train

In [ ]:
trainer.push_to_hub()